# Reproducción del proyecto en Google Colab

**Subocupación de buses intermunicipales en Colombia y su efecto en la huella de carbono (enero de 2025 a marzo de 2026).**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KarollPintoC/Proyecto_Analitiaca_Datos_transporte/blob/main/notebooks/Colab_Reproduccion.ipynb)

Este cuaderno clona el repositorio y ejecuta los cuadernos que trabajan sobre el archivo parquet versionado, sin Docker ni PostgreSQL:

| Fase | Cuaderno | En Colab |
|---|---|---|
| 1 | `Fase1_ETL.ipynb` (descarga de la API y carga a PostgreSQL) | No: requiere Docker; su salida `datos/despachos_enriquecidos.parquet` está en el repositorio |
| 2 | `Fase2_EDA.ipynb` | Sí |
| 3 | `Fase3_Modelado_Ambiental.ipynb` | Sí |
| 4 | `Fase4_Modelo_Dimensional.ipynb` (modelo en estrella en PostgreSQL) | No: requiere Docker; su validación está en `results/tables/fase4_validacion_modelo_estrella.csv` |
| 5 | `Fase5_Modelado_Predictivo.ipynb` | Opcional (1 a 2 horas; modo rápido para verificar) |

Al final se comparan las tablas regeneradas con las versionadas en el repositorio.

## 1. Clonar el repositorio e instalar las dependencias

In [ ]:
import os, subprocess
if not os.path.exists("/content/proyecto"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/KarollPintoC/Proyecto_Analitiaca_Datos_transporte.git", "/content/proyecto"], check=True)
os.chdir("/content/proyecto/notebooks")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
%pip install -q polars holidays optuna lightgbm psycopg2-binary seaborn nbconvert

## 2. Parámetros de la ejecución

- `EJECUTAR_FASE5`: la corrida completa de la Fase 5 tarda de 1 a 2 horas.
- `MODO_RAPIDO_FASE5`: con `True` se ejecuta con pocas pruebas de Optuna, solo para verificar que el cuaderno corre. **Sus métricas no se reportan**: la corrida válida es con `False`.

In [ ]:
EJECUTAR_FASE5 = False
MODO_RAPIDO_FASE5 = True

## 3. Ejecución de los cuadernos

In [ ]:
import time, nbformat
from nbconvert.preprocessors import ExecutePreprocessor

def ejecutar(nombre, reemplazos=None):
    """Ejecuta un cuaderno completo y lo guarda con sus salidas."""
    inicio = time.time()
    nb = nbformat.read(nombre, as_version=4)
    for celda in nb.cells:
        for viejo, nuevo in (reemplazos or {}).items():
            celda.source = celda.source.replace(viejo, nuevo)
    ExecutePreprocessor(timeout=-1, kernel_name="python3").preprocess(nb, {"metadata": {"path": "."}})
    nbformat.write(nb, nombre)
    print(f"{nombre}: ejecutado en {(time.time() - inicio) / 60:.1f} min")

ejecutar("Fase2_EDA.ipynb")
ejecutar("Fase3_Modelado_Ambiental.ipynb")
if EJECUTAR_FASE5:
    ejecutar("Fase5_Modelado_Predictivo.ipynb",
             {"MODO_RAPIDO = False": f"MODO_RAPIDO = {MODO_RAPIDO_FASE5}"})

## 4. Verificación de la reproducibilidad

Se comparan las tablas generadas en esta ejecución con las versionadas en el repositorio. Las de las Fases 2 y 3 deben coincidir; si la Fase 5 se ejecutó en modo rápido, sus tablas cambian por diseño.

In [ ]:
import pandas as pd
from io import StringIO

def version_repositorio(ruta):
    texto = subprocess.run(["git", "show", f"HEAD:{ruta}"], capture_output=True, text=True, cwd="/content/proyecto").stdout
    return pd.read_csv(StringIO(texto))

filas = []
for prefijo in ["eda_", "fase3_"] + (["fase5_"] if EJECUTAR_FASE5 else []):
    for archivo in sorted(os.listdir("../results/tables")):
        if archivo.startswith(prefijo):
            ruta = f"results/tables/{archivo}"
            nuevo, versionado = pd.read_csv(f"../{ruta}"), version_repositorio(ruta)
            try:
                pd.testing.assert_frame_equal(nuevo, versionado, check_exact=False, rtol=1e-6, check_dtype=False)
                igual = True
            except AssertionError:
                igual = False
            filas.append({"tabla": archivo, "coincide": igual})
verificacion = pd.DataFrame(filas)
display(verificacion)
print(f"Tablas que coinciden: {verificacion['coincide'].sum()} de {len(verificacion)}")

## 5. Resultados principales

In [ ]:
display(pd.read_csv("../results/tables/fase3_01_kpi_linea_base.csv"))
display(pd.read_csv("../results/tables/fase3_04_sensibilidad_capacidad.csv"))
display(pd.read_csv("../results/tables/fase5_07_comparacion_modelos.csv"))
display(pd.read_csv("../results/tables/fase5_09_ablacion.csv"))

In [ ]:
from IPython.display import Image
for figura in ["fig_f3_01_contraste_real_ideal.png", "fig_f3_03_intensidad_mensual.png",
               "fig_f5_01_comparacion_modelos.png", "powerbi_01_resumen_ejecutivo.png"]:
    display(Image(f"../results/figures/{figura}", width=900))